In [1]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer
from datasets import load_dataset, load_metric

D:\Anaconda\envs\FineTuning\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. 加载 Qwen2.5-0.5B 预训练模型和分词器
model_name = "E:\Transformers\Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # 设置 padding token
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)  # 2分类任务
model.config.pad_token_id = tokenizer.pad_token_id

<>:2: SyntaxWarning: invalid escape sequence '\T'
<>:2: SyntaxWarning: invalid escape sequence '\T'
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_6512\2618749820.py:2: SyntaxWarning: invalid escape sequence '\T'
  model_name = "E:\Transformers\Qwen2.5-0.5B-Instruct"
Some weights of Qwen2ForSequenceClassification were not initialized from the model checkpoint at E:\Transformers\Qwen2.5-0.5B-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [3]:
# 2. 加载并预处理数据集
dataset = load_dataset("imdb", split='train[:1%]')  # 只加载10%的训练数据
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

tokenized_datasets = dataset.map(preprocess_function, batched=True)

tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

train_dataset = tokenized_datasets
test_dataset = load_dataset("imdb", split='test[:1%]').map(preprocess_function, batched=True)
test_dataset = test_dataset.rename_column("label", "labels")
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map: 100%|██████████| 250/250 [00:00<00:00, 5132.38 examples/s]


In [4]:
# 3. 设置训练参数
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    load_best_model_at_end=True,
)

D:\Anaconda\envs\FineTuning\Lib\site-packages\transformers\training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [5]:
# 4. 评估指标
metric = load_metric("accuracy", trust_remote_code=True)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = torch.argmax(torch.tensor(logits), dim=-1)
    return metric.compute(predictions=predictions, references=labels)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_6512\1498804354.py:2: FutureWarning: load_metric is deprecated and will be removed in the next major version of datasets. Use 'evaluate.load' instead, from the new library 🤗 Evaluate: https://huggingface.co/docs/evaluate
  metric = load_metric("accuracy", trust_remote_code=True)


In [6]:
from transformers import DataCollatorWithPadding

# 5. 训练
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.000000,0.000000,1.000000


TrainOutput(global_step=125, training_loss=5.4870979001862e-06, metrics={'train_runtime': 73.8264, 'train_samples_per_second': 3.386, 'train_steps_per_second': 1.693, 'total_flos': 274867126272000.0, 'train_loss': 5.4870979001862e-06, 'epoch': 1.0})

In [7]:
# 6. 评估
trainer.evaluate()

{'eval_loss': 0.0,
 'eval_accuracy': 1.0,
 'eval_runtime': 17.0957,
 'eval_samples_per_second': 14.624,
 'eval_steps_per_second': 7.312,
 'epoch': 1.0}

In [8]:
# 7. 保存模型
tokenizer.save_pretrained("./qwen2.5-classifier")
model.save_pretrained("./qwen2.5-classifier")